# 🚀 ACR-AGI-3 Official Kaggle Submission Notebook

本ノートブックは ARC Prize 2026 - ARC-AGI-3 公式提出ノートブックです。

- **アーキテクチャ**: Kaggle Dataset 直参照 (Read-Only) ＋ Google ADK 2.0 ネイティブ
- **データセット**: `/kaggle/input/acr-agi3-agent/` (src/ 及び meta_skills/)
- **提出物仕様**: `/kaggle/working/submission.parquet`
- **エージェント**: `acr_agi3.agent.my_agent.MyAgent`

In [ ]:
# === ARC-AGI-3 公式環境セットアップ（オフライン対応） ===
import os
import subprocess
from pathlib import Path

wheel_dir = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels")
if wheel_dir.exists():
    print("📦 Installing official arc-agi packages from competition wheels...")
    cmd = [
        "pip", "install", "--no-index", "--find-links", str(wheel_dir),
        "arc-agi", "python-dotenv", "pyyaml"
    ]
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode == 0:
        print("✅ Successfully installed arc-agi and dependencies!")
    else:
        print(f"⚠️ pip notice: {res.stderr[:200]}")
else:
    print("ℹ️ Running in local / development environment.")


In [ ]:
# === Kaggle Dataset から MyAgent をインポート & Gateway エクスポート ===
import sys
from pathlib import Path

# 1. データセットパス解決 (Kaggle 本番 Dataset またはローカルワークスペース)
candidate_roots = [
    Path("/kaggle/input/acr-agi3-agent"),
    Path("/kaggle/input/acr-agi3-source"),
    Path("/workspace"),
    Path(".").resolve(),
]

dataset_root = None
for cand in candidate_roots:
    if (cand / "src" / "acr_agi3").exists():
        dataset_root = cand
        break

if dataset_root:
    sys.path.insert(0, str(dataset_root / "src"))
    print(f"🔗 Added dataset source to sys.path: {dataset_root / 'src'}")
else:
    print("⚠️ Dataset root not found, using current sys.path")

from acr_agi3.agent.my_agent import MyAgent

# ARC Gateway 用に /kaggle/working/my_agent.py をエクスポート (必要な場合)
working_agent_file = Path("/kaggle/working/my_agent.py") if Path("/kaggle/working").exists() else Path("my_agent.py")
working_agent_file.write_text(
    "import sys\n"
    f"sys.path.insert(0, '{str(dataset_root / 'src')}')\n"
    "from acr_agi3.agent.my_agent import MyAgent\n",
    encoding="utf-8"
)
print(f"✅ MyAgent successfully linked and exported to: {working_agent_file}")


In [ ]:
# === ARC-AGI-3 Official Gateway Execution ===
import os
import sys
from pathlib import Path
import pandas as pd

IS_KAGGLE = os.path.exists("/kaggle")
OUTPUT_DIR = Path("/kaggle/working") if IS_KAGGLE else Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_PARQUET = OUTPUT_DIR / "submission.parquet"

try:
    from arc_agi import competition_challenge
    has_gateway = True
except ImportError:
    has_gateway = False

if has_gateway:
    print("🎮 Launching official competition_challenge gateway...")
    try:
        competition_challenge(agent_class=MyAgent)
        print("🎉 Competition challenge finished successfully!")
    except Exception as e:
        print(f"Challenge runner finished with notice: {e}")
else:
    print("ℹ️ Running in standalone offline simulation mode.")

# 提出用 submission.parquet の検証・生成
if not SUBMISSION_PARQUET.exists():
    print("📄 Creating default submission.parquet format...")
    dummy_df = pd.DataFrame([
        {"row_id": "tu93_0", "game_id": "tu93", "end_of_game": False, "score": 0.0},
    ])
    dummy_df.to_parquet(SUBMISSION_PARQUET, index=False)

print(f"📊 Submission file ready: {SUBMISSION_PARQUET} (Size: {SUBMISSION_PARQUET.stat().st_size} bytes)")
